# Dia 16 — Perguntando ao mundo 🔎

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-16-automacao-de-busca](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-16-automacao-de-busca)

## 🎬 Missão

> Segunda-feira, fim de tarde. Bento está com dezessete abas abertas no
> navegador, todas com buscas sobre a PixVelox.
>
> — Copiar e colar resultado de busca é o jeito mais caro de gastar a vida —
> resmunga a Mestra Iara, fechando o notebook dele. — O terminal pergunta ao
> mundo por você, e guarda a resposta num arquivo.
>
> Ela deixa dois arquivos na pasta do caso: uma busca do DuckDuckGo e uma
> exportação do **Netlas**, um buscador que não procura páginas, e sim
> **servidores**.
> — Um dos sites dessa busca não é vitrine. É bastidor. Ache, e veja o que
> ele expõe porta por porta.

**Sua tarefa:** processar os resultados salvos em `desafio/`, identificar o
host da infraestrutura escondida e montar a flag `VAZANTE{...}` a partir dos
títulos dos servidores dele.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### ddgr: DuckDuckGo no terminal

```text
ddgr faz buscas no DuckDuckGo e mostra o resultado em texto.
  -n 10        quantos resultados
  --np         "no prompt": busca e sai (sem o menu interativo)
  -C           sem cores (bom para salvar em arquivo)
  -x           mostra a URL completa de cada resultado
Redirecionar com > salva o resultado para trabalhar depois.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Buscando com ddgr"
precisa ddgr
if online && ddgr -n 10 --np -C osint > busca_osint.txt 2>/dev/null && [ -s busca_osint.txt ]; then
  head -n 12 busca_osint.txt
  rm busca_osint.txt
else
  alerta "offline (ou DuckDuckGo bloqueou): usando cópia salva"
  head -n 12 desafio/busca_pixvelox.txt
fi

### Extraindo só os sites com grep -o

```text
Na saída de texto, o ddgr mostra o site de cada resultado entre colchetes.
grep -o '\[.*]' pega só esse pedaço ([ precisa de \ porque é especial).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Quais sites apareceram?"
grep -o '\[.*]' desafio/busca_pixvelox.txt
echo "--- sem colchetes, contando ---"
grep -o '\[.*]' desafio/busca_pixvelox.txt | tr -d '[]' | sort | uniq -c | sort -rn

### ddgr --json + jq: resultados estruturados

```text
Com --json o ddgr devolve uma LISTA de objetos {title, url, abstract}.
jq (Dia 14) tira o que você quiser: .[] percorre a lista; -r tira as aspas.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Resultados em JSON"
precisa jq
if online && ddgr -n 5 --json pixvelox > resultados.json 2>/dev/null && [ -s resultados.json ]; then
  jq -r '.[] | .url' resultados.json
  rm resultados.json
else
  alerta "offline: usando cópia salva"
  jq -r '.[] | .url' desafio/busca_pixvelox.json
fi
echo "--- resultados cujo resumo fala em 'código' ---"
jq -r '.[] | select(.abstract | test("código")) | .title' desafio/busca_pixvelox.json

### Operadores de busca (dorks)

```text
Buscadores entendem operadores que refinam a pergunta:
  site:dominio      só resultados daquele site
  filetype:pdf      só arquivos daquele tipo (xlsx, docx, csv...)
  intitle:palavra   a palavra no TÍTULO da página
  inurl:palavra     a palavra no ENDEREÇO
  "frase exata"     entre aspas, exatamente assim
  -palavra          exclui resultados com a palavra
Uma lista de dorks prontas está em desafio/dorks.txt.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Dorks do caso"
cat desafio/dorks.txt
dica "No ddgr: ddgr --np -w social.example pixvelox   (-w = site:)"

### Montando URLs de busca (urlencode)

```text
Toda busca vira uma URL. Espaços, aspas e dois-pontos precisam ser
"codificados" (urlencode): espaço vira %20, aspas viram %22, : vira %3A...
O jq faz isso com o filtro @uri. Com a URL pronta, o curl (ou você no
navegador) abre a busca — e um laço gera dezenas de buscas de uma vez.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Gerando URLs de busca para cada dork"
precisa jq
while IFS= read -r dork; do
  q=$(jq -rn --arg q "$dork" '$q | @uri')          # urlencode do texto
  echo "https://duckduckgo.com/html/?q=$q"
done < desafio/dorks.txt
dica "Abra no navegador ou use: curl -s -A 'Mozilla/5.0' 'URL' — com moderação."

### Automatizando várias buscas (com pausa!)

```text
Para muitas buscas: laço + sleep entre elas, para não ser bloqueado.
O ddgr aceita --proxy se você precisar trocar de saída de rede.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Laço de buscas (simulado offline)"
precisa ddgr
while IFS= read -r dork; do
  arquivo="busca_$(echo "$dork" | tr -c '[:alnum:]' '_' | cut -c1-30).txt"
  if online; then
    mostrar "ddgr -n 10 --np -C '$dork' > $arquivo ; sleep 3"
  else
    mostrar "(offline) ddgr -n 10 --np -C '$dork' > $arquivo"
  fi
done < desafio/dorks.txt

### Netlas: o buscador de servidores (opcional, precisa de API key)

```text
Buscadores "de IP" (Netlas, Shodan, Censys) não indexam páginas, e sim
SERVIDORES: cada IP, porta, título da página, certificado...
Plano gratuito do Netlas: crie conta em app.netlas.io e copie sua API key.
  pip install netlas
  netlas savekey SUA_CHAVE
  netlas download -c 20 "http.body:*pixvelox*" > netlas_resultados.json
Outros filtros: http.title:  domain:  port:  geo.city:  geo.country:  ip:
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Netlas"
precisa_pip netlas netlas || true
mostrar "netlas savekey SUA_CHAVE"
mostrar 'netlas download -c 20 "http.body:*pixvelox*" > netlas_resultados.json'
if online && [ -n "${NETLAS_API_KEY:-}" ] && command -v netlas >/dev/null 2>&1; then
  netlas download -a "$NETLAS_API_KEY" -c 5 'http.title:"Example Domain"' > netlas_teste.json \
    && jq -r '.data.host' netlas_teste.json
  rm -f netlas_teste.json
else
  dica "Sem NETLAS_API_KEY (ou offline): vamos usar a cópia salva em desafio/netlas_resultados.json"
fi

### jq nos resultados do Netlas

```text
O netlas download grava UM objeto JSON por linha (formato "JSON Lines").
O jq processa linha por linha naturalmente.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Títulos das páginas encontradas pelo Netlas"
precisa jq
jq .data.http.title desafio/netlas_resultados.json
echo "--- IP, porta e host, em colunas ---"
jq -r '[.data.ip, .data.port, .data.host] | @tsv' desafio/netlas_resultados.json
echo "--- quantos servidores por IP ---"
jq -r '.data.ip' desafio/netlas_resultados.json | sort | uniq -c | sort -rn

### Cruzando as duas buscas

```text
Um site apareceu no DuckDuckGo. Será que ele aparece também no Netlas?
comm (Dia 10) mostra o que é comum a duas listas ORDENADAS.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Sites que estão nas duas buscas"
precisa jq
grep -o '\[.*]' desafio/busca_pixvelox.txt | tr -d '[]' | sort -u > ddg.txt
jq -r '.data.host' desafio/netlas_resultados.json | sort -u > netlas.txt
comm -12 ddg.txt netlas.txt
rm ddg.txt netlas.txt
dica "Algum desses não é um site 'de vitrine'... olhe as portas dele."

### Versão brazuca: buscando em diários oficiais (Querido Diário)

```text
O Querido Diário (Open Knowledge Brasil) indexa diários oficiais de
municípios e tem API pública, sem chave. Montamos a URL com urlencode.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && source "$L4O_HOME/lib/comum.sh"
titulo "Querido Diário"
precisa jq
termo="licitação pix"
q=$(jq -rn --arg q "$termo" '$q | @uri')
URL="https://api.queridodiario.ok.org.br/gazettes?querystring=$q&size=3&excerpt_size=200&number_of_excerpts=1"
echo "$URL"
if online && curl -s -m 20 "$URL" -o qd.json && jq -e '.gazettes' qd.json >/dev/null 2>&1; then
  jq -r '.total_gazettes as $t | "total de diários: \($t)"' qd.json
  jq -r '.gazettes[] | "\(.date)  \(.territory_name) (\(.state_code))  \(.url)"' qd.json
else
  alerta "offline (ou API indisponível): veja a URL acima e abra quando tiver internet"
fi
rm -f qd.json

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-16-automacao-de-busca" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)